# ЛР-03: Бюджет полевой связи

## Worked example: military 01

Это полностью разобранный example по duality и анализу чувствительности. Он показывает полный цикл от прямой модели до сценарных пересчётов.

## 1. Исходный кейс

Разобранный duality-кейс на планировании систем полевой связи.

### Ограничения ресурсов

| Ресурс | Лимит |
| --- | --- |
| Бюджет | 92 |
| Специалисты | 55 |
| Технические слоты | 46 |

### Программы

| Программа | Эффект | Бюджет | Трудозатраты | Операционная ёмкость |
| --- | --- | --- | --- | --- |
| Мобильные ретрансляторы | 84 | 34 | 16 | 15 |
| Защищённые терминалы | 79 | 28 | 13 | 12 |
| Резерв кабельных комплектов | 68 | 18 | 9 | 8 |
| Узлы командной связи | 96 | 46 | 24 | 20 |

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

def solve_primal(effects, A_ub, b_ub):
    result = linprog(-effects, A_ub=A_ub, b_ub=b_ub, bounds=[(0, 1)] * len(effects), method='highs')
    if not result.success:
        raise RuntimeError(result.message)
    shadow_prices = -result.ineqlin.marginals
    return result, shadow_prices

def solve_dual(effects, A_ub, b_ub):
    m, n = A_ub.shape
    c_dual = np.concatenate([b_ub, np.ones(n)])
    A_dual = -np.hstack([A_ub.T, np.eye(n)])
    b_dual = -effects
    result = linprog(c_dual, A_ub=A_dual, b_ub=b_dual, bounds=[(0, None)] * (m + n), method='highs')
    if not result.success:
        raise RuntimeError(result.message)
    return result

def rerun_with_resource_change(effects, A_ub, b_ub, resource_index, delta):
    new_b = b_ub.copy()
    new_b[resource_index] += delta
    result, _ = solve_primal(effects, A_ub, new_b)
    return new_b, result


In [2]:
program_names = ['Мобильные ретрансляторы', 'Защищённые терминалы', 'Резерв кабельных комплектов', 'Узлы командной связи']
resource_names = ['Бюджет', 'Специалисты', 'Технические слоты']
effects = np.array([84, 79, 68, 96], dtype=float)
A_ub = np.array([[34, 28, 18, 46], [16, 13, 9, 24], [15, 12, 8, 20]], dtype=float)
b_ub = np.array([92, 55, 46], dtype=float)

primal_result, shadow_prices = solve_primal(effects, A_ub, b_ub)
dual_result = solve_dual(effects, A_ub, b_ub)

plan_df = pd.DataFrame({
    'программа': program_names,
    'x*': np.round(primal_result.x, 4),
    'эффект на единицу': effects,
})
resources_df = pd.DataFrame({
    'ресурс': resource_names,
    'лимит': b_ub,
    'slack': np.round(primal_result.slack, 4),
    'shadow_price': np.round(shadow_prices, 4),
    'binding': np.isclose(primal_result.slack, 0.0),
})

print('Оптимальный эффект (primal):', round(-primal_result.fun, 4))
print('Оптимальное значение dual:', round(dual_result.fun, 4))
print()
print('Оптимальный план:')
display(plan_df)
print('Ресурсный разбор:')
display(resources_df)


Оптимальный эффект (primal): 256.0435
Оптимальное значение dual: 256.0435

Оптимальный план:


,программа,x*,эффект на единицу
0,Мобильные ретрансляторы,1.0000,84.0
1,Защищённые терминалы,1.0000,79.0
2,Резерв кабельных комплектов,1.0000,68.0
3,Узлы командной связи,0.2609,96.0


Ресурсный разбор:


,ресурс,лимит,slack,shadow_price,binding
0,Бюджет,92.0,0.0000,2.087,True
1,Специалисты,55.0,10.7391,0.000,False
2,Технические слоты,46.0,5.7826,0.000,False


In [3]:
scenario_rows = []
for label, resource_index, delta in [
    ('Бюджет +2', 0, 2),
    ('Специалисты +2', 1, 2),
]:
    new_b, new_result = rerun_with_resource_change(effects, A_ub, b_ub, resource_index, delta)
    predicted = shadow_prices[resource_index] * delta
    actual = (-new_result.fun) - (-primal_result.fun)
    scenario_rows.append({
        'сценарий': label,
        'ресурс': resource_names[resource_index],
        'delta': delta,
        'прогноз по shadow price': round(predicted, 4),
        'факт после пересчёта': round(actual, 4),
        'разница': round(actual - predicted, 4),
    })

scenario_df = pd.DataFrame(scenario_rows)
display(scenario_df)


,сценарий,ресурс,delta,прогноз по shadow price,факт после пересчёта,разница
0,Бюджет +2,Бюджет,2,4.1739,4.1739,-0.0
1,Специалисты +2,Специалисты,2,0.0000,0.0000,0.0


In [4]:
objective_label = 'Ретрансляторы +5 к эффекту'
objective_index = 0
objective_delta = 5

new_effects = effects.copy()
new_effects[objective_index] += objective_delta
objective_result, _ = solve_primal(new_effects, A_ub, b_ub)
objective_df = pd.DataFrame({
    'сценарий': [objective_label],
    'новый оптимальный эффект': [round(-objective_result.fun, 4)],
    'изменение эффекта': [round((-objective_result.fun) - (-primal_result.fun), 4)],
    'новое значение программы': [round(objective_result.x[objective_index], 4)],
})
display(objective_df)


,сценарий,новый оптимальный эффект,изменение эффекта,новое значение программы
0,Ретрансляторы +5 к эффекту,261.0435,5.0,1.0


## 2. Что важно проговорить в выводе

- какие ограничения оказались binding и почему именно они держат оптимум;
- какой ресурс имеет наибольшую теневую цену и что это означает содержательно;
- насколько хорошо совпал прогноз по shadow price с фактическим пересчётом;
- меняется ли структура плана при небольших изменениях `b` и `c`.